In [1]:
import numpy as np
import time
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

# Reproducibilidad de los resultados
np.random.seed(42)

# -------------------------------------------------
# 1. Generar 50.000 ejemplos de multiplicacion
# -------------------------------------------------

# Cada matriz tiene cuatro elementos entre -20 y 20
A = np.random.randint(-20, 21, size=(50000, 2, 2))
B = np.random.randint(-20, 21, size=(50000, 2, 2))

# Calcular las respuestas correctas
C = np.matmul(A, B)

# La red recibe 8 entradas y predice 4 salidas
X = np.concatenate(
    (A.reshape(50000, 4), B.reshape(50000, 4)),
    axis=1
)

Y = C.reshape(50000, 4)

# Normalizar entradas y salidas para facilitar
# el entrenamiento de la red neuronal
X = X / 20.0
Y = Y / 800.0

# Separar datos de entrenamiento y prueba
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=10, random_state=42
)

# -------------------------------------------------
# 2. Crear y entrenar el modelo
# -------------------------------------------------

modelo = MLPRegressor(
    hidden_layer_sizes=(64, 64),
    activation="relu",
    solver="adam",
    max_iter=300,
    random_state=42,
    early_stopping=True,
    n_iter_no_change=15
)

inicio_entrenamiento = time.perf_counter()
modelo.fit(X_train, Y_train)
fin_entrenamiento = time.perf_counter()

print("Tiempo de entrenamiento:",
      fin_entrenamiento - inicio_entrenamiento, "segundos")

# -------------------------------------------------
# 3. Evaluar 10 ejemplos
# -------------------------------------------------

inicio_prediccion = time.perf_counter()
predicciones = modelo.predict(X_test) * 800.0
fin_prediccion = time.perf_counter()

# Recuperar las matrices originales
A_test = (X_test[:, :4] * 20).astype(int).reshape(-1, 2, 2)
B_test = (X_test[:, 4:] * 20).astype(int).reshape(-1, 2, 2)

print("\nRESULTADOS DE LOS 10 EJEMPLOS")

for i in range(10):
    resultado_analitico = A_test[i] @ B_test[i]
    resultado_modelo = np.rint(predicciones[i]).astype(int).reshape(2, 2)

    print(f"\nEjemplo {i + 1}")
    print("Matriz A:\n", A_test[i])
    print("Matriz B:\n", B_test[i])
    print("Resultado analitico:\n", resultado_analitico)
    print("Prediccion del modelo:\n", resultado_modelo)
    print("Coincidencia exacta:",
          np.array_equal(resultado_analitico, resultado_modelo))

# -------------------------------------------------
# 4. Medir el error del modelo
# -------------------------------------------------

Y_test_original = Y_test * 800.0

error = mean_absolute_error(
    Y_test_original.flatten(),
    predicciones.flatten()
)

print("\nError absoluto medio:", error)
print("Tiempo de prediccion de 10 ejemplos:",
      fin_prediccion - inicio_prediccion, "segundos")

# -------------------------------------------------
# 5. Medir el costo de la operacion tradicional
# -------------------------------------------------

inicio_tradicional = time.perf_counter()

for i in range(10):
    resultado = A_test[i] @ B_test[i]

fin_tradicional = time.perf_counter()

print("Tiempo analitico para 10 ejemplos:",
      fin_tradicional - inicio_tradicional, "segundos")

Tiempo de entrenamiento: 71.84226853999999 segundos

RESULTADOS DE LOS 10 EJEMPLOS

Ejemplo 1
Matriz A:
 [[11 -3]
 [ 6 -7]]
Matriz B:
 [[ 10  16]
 [  0 -11]]
Resultado analitico:
 [[110 209]
 [ 60 173]]
Prediccion del modelo:
 [[121 214]
 [ 67 186]]
Coincidencia exacta: False

Ejemplo 2
Matriz A:
 [[  2 -18]
 [ 14  18]]
Matriz B:
 [[ 15   6]
 [-11  -9]]
Resultado analitico:
 [[228 174]
 [ 12 -78]]
Prediccion del modelo:
 [[225 184]
 [ 24 -85]]
Coincidencia exacta: False

Ejemplo 3
Matriz A:
 [[18 -4]
 [18  1]]
Matriz B:
 [[ 20 -10]
 [  1  -8]]
Resultado analitico:
 [[ 356 -148]
 [ 361 -188]]
Prediccion del modelo:
 [[ 338 -165]
 [ 344 -194]]
Coincidencia exacta: False

Ejemplo 4
Matriz A:
 [[2 8]
 [9 4]]
Matriz B:
 [[-15  15]
 [  2  -9]]
Resultado analitico:
 [[ -14  -42]
 [-127   99]]
Prediccion del modelo:
 [[  -8  -43]
 [-134   97]]
Coincidencia exacta: False

Ejemplo 5
Matriz A:
 [[ 4 12]
 [ 7 11]]
Matriz B:
 [[-12  -8]
 [  9  -1]]
Resultado analitico:
 [[ 60 -44]
 [ 15 -67]]
Predi